# NAS - Optuna

- **Authored by:** Matheus Ferreira Silva 
- **GitHub:**: https://github.com/MatheusFS-dev

## 1. Setup and Configuration

### 1.1. Imports

In [1]:
from _imports import *
from araras.ml.optuna.model_tools import plot_model_param_distribution

2025-08-05 14:24:23.657545: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-05 14:24:23.671713: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754414663.688480 1014246 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754414663.693414 1014246 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-08-05 14:24:23.710223: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

### 1.2. Policy

In [2]:
POLICY = mixed_precision.Policy("mixed_float16")
mixed_precision.set_global_policy(POLICY)

### 1.3. Constants

In [3]:
DATA_SEED = 99
TRAIN_SEED = 111

## 2. Data Loading and Preprocessing

In [4]:
(
    s008_coord_input,
    s008_lidar_input,
    s008_y_train,
    s009_coord_input,
    s009_lidar_input,
    s009_y,
) = load_dataset_sparse_labels()

/home/matheus/src/RayWise/src/_load_dataset.py:42: ComplexWarning: Casting complex values to real discards the imaginary part
  s008_y_train = s008_y_train.astype(np.float32)
/home/matheus/src/RayWise/src/_load_dataset.py:65: ComplexWarning: Casting complex values to real discards the imaginary part
  s008_y_val = s008_y_val.astype(np.float32)


Shape before conversion: (9234, 8, 32)
Shape after conversion: (9234,)
y_train shape: (9234,)
coord_input shape: (9234, 2)
lidar_input shape: (9234, 20, 200, 10)
Shape before conversion: (1960, 8, 32)
Shape after conversion: (1960,)
y_val shape: (1960,)
coord_input_val shape: (1960, 2)
lidar_input_val shape: (1960, 20, 200, 10)
y_train shape: (11194,)
coord_input shape: (11194, 2)
lidar_input shape: (11194, 20, 200, 10)
Shape before conversion: (9638, 8, 32)
Shape after conversion: (9638,)
y shape: (9638,)
coord_input shape: (9638, 2)
lidar_input shape: (9638, 20, 200, 10)


/home/matheus/src/RayWise/src/_load_dataset.py:101: ComplexWarning: Casting complex values to real discards the imaginary part
  s009_y = s009_y.astype(np.float32)


In [5]:
(
    x_s008_lidar_train,
    x_s008_lidar_val,
    x_s008_coord_train,
    x_s008_coord_val,
    y_s008_train,
    y_val,
) = train_test_split(
    s008_lidar_input,
    s008_coord_input,
    s008_y_train,
    test_size=0.2,
    random_state=DATA_SEED,
    shuffle=True,
)

(
    x_s009_lidar_test,
    x_s009_lidar_val,
    x_s009_coord_test,
    x_s009_coord_val,
    y_s009_test,
    y_s009_val,
) = train_test_split(
    s009_lidar_input,
    s009_coord_input,
    s009_y,
    test_size=0.2,
    random_state=DATA_SEED,
    shuffle=True,
)

x_lidar_train = x_s008_lidar_train
x_coord_train = x_s008_coord_train
y_train = y_s008_train

x_lidar_val = np.concatenate((x_s008_lidar_val, x_s009_lidar_val), axis=0)
x_coord_val = np.concatenate((x_s008_coord_val, x_s009_coord_val), axis=0)
y_val = np.concatenate((y_val, y_s009_val), axis=0)

x_lidar_test = x_s009_lidar_test
x_coord_test = x_s009_coord_test
y_test = y_s009_test

## 3. Hyperparameters

In [6]:
kparams = KParams.default()
kparams.learning_rate = 7e-5

I0000 00:00:1754414666.793255 1014246 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5115 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3070 Ti, pci bus id: 0000:b3:00.0, compute capability: 8.6


## 4. Model Architectures

In [7]:
def build_model(trial: optuna.Trial, kparams: dict, show_summary: bool = True) -> tf.keras.Model:
    from spektral.layers import GraphMasking, GlobalAvgPool

    # ———————————————————————————————————————————————————————————————————————————— #
    #                              Model Construction                              #
    # ———————————————————————————————————————————————————————————————————————————— #

    # ———————————————————————————————— LiDAR Input ——————————————————————————————— #
    x_lidar_input = layers.Input(shape=(20, 200, 10), name="lidar_input")

    # Inline one-hot encoding of semantic values
    one_hot_lidar = layers.Lambda(
        lambda x: tf.concat(
            [
                # “Is there a BS anywhere in the 10 channels?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, -2), axis=-1, keepdims=True), tf.float32),
                # “Vehicle?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, -1), axis=-1, keepdims=True), tf.float32),
                # “Obstacle?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, 1), axis=-1, keepdims=True), tf.float32),
                # “Free?” → 1 channel (all channels zero)
                tf.cast(tf.reduce_all(tf.equal(x, 0), axis=-1, keepdims=True), tf.float32),
            ],
            axis=-1,
        ),
        #! Lambda has deserialization issues, so providing the output shape is necessary
        output_shape=(20, 200, 4),
        name="lidar_transform_to_one_hot",
    )(x_lidar_input)
    # -> (batch, 20, 200, 4)

    # Flatten the 20×200 grid into a 4000-length sequence with the 4 channels
    x_lidar_flat: layers.Layer = layers.Reshape((20 * 200, 4), name="lidar_flatten_4_channels")(one_hot_lidar)

    # ———————————————————————————————— GPS Input ———————————————————————————————— #
    # Input for coordinate data (e.g., shape: (2,))
    x_coord_input = layers.Input(shape=(2,), name="coord_input")

    # Turn (batch,2) → (batch,1,2) → tile to (batch,4000,2)'
    x_coord: layers.Layer = layers.Lambda(
        lambda x: tf.tile(tf.expand_dims(x, axis=1), [1, 20 * 200, 1]),
        #! Lambda has deserialization issues, so providing the output shape is necessary
        output_shape=(20 * 200, 2),
        name="coord_tile_flat",
    )(x_coord_input)

    # ————————————————————————————— Combine Branches ————————————————————————————— #
    # Fuse channels:  (batch,4000,4) + (batch,4000,2) → (batch,4000,6)
    combined = layers.Concatenate(axis=-1, name="combine_lidar_coord")([x_lidar_flat, x_coord])

    # ———————————————————————————————— Initializer ———————————————————————————————— #
    initializer = tf.keras.initializers.GlorotUniform(
        seed=TRAIN_SEED,
    )

    # ———————————————————————————————————— GNN ——————————————————————————————————— #
    A = build_knn_adjacency(rows=20, cols=200, k=12)

    x_graph, a_graph = GraphMasking()([combined, A])

    gnn1 = build_cheb(
        trial=trial,
        kparams=None,
        x=x_graph,  # Use x_graph only for the first layer
        a_graph=a_graph,
        name_prefix="cheb_1",
        units_range=380,
        K_range=2,
        dropout_rate_range=0.25,
        activation="relu",
        kernel_initializer=initializer,
    )

    gnn2 = build_cheb(
        trial=trial,
        kparams=None,
        x=gnn1,
        a_graph=a_graph,
        name_prefix="cheb_2",
        units_range=370,
        K_range=2,
        dropout_rate_range=0.05,
        activation="None",
        kernel_initializer=initializer,
    )

    gnn3 = build_cheb(
        trial=trial,
        kparams=None,
        x=gnn2,
        a_graph=a_graph,
        name_prefix="cheb_3",
        units_range=270,
        K_range=2,
        dropout_rate_range=0.5,
        activation="elu",
        kernel_initializer=initializer,
    )

    # GNN output shape: (batch, nodes, channels)
    x = trial_skip_3d_tensors(
        trial=trial,
        layers_list=[gnn1, gnn2, gnn3],
        nickname=["gnn1", "gnn2", "gnn3"],
        axis_to_concat=-1,
        strategy="any",
        name_prefix="skip_gnn",
        merge_mode="concat",
        projection_mode="resize",
        verbose=0,
    )

    # Global pooling layer to reduce the graph to a fixed-size vector
    #! The GlobalAvgPool layer from Spektral causes an error with the skip connection,
    #! it adds a singleton channel dimension to 2d tensors. Probably due to the masking.
    x = GlobalAvgPool(name="global_avg_pool")(x)

    warnings.filterwarnings("ignore", message=".*Flatten.*mask.*support masking.*")

    # Flatten to remove the singleton dimension
    x = layers.Flatten(name="flatten_gnn_output")(x)

    # ———————————————————————————— Extra dense layers ———————————————————————————— #
    dnn1 = build_dnn(
        trial=trial,
        kparams=None,
        x=x,
        name_prefix="dense_1",
        activation="elu",
        units_range=250,
        dropout_rate_range=0.0,
        kernel_initializer=initializer,
    )

    dnn2 = build_dnn(
        trial=trial,
        kparams=None,
        x=dnn1,
        name_prefix="dense_2",
        activation="tanh",
        units_range=550,
        dropout_rate_range=0.0,
        kernel_initializer=initializer,
    )

    dnn3 = build_dnn(
        trial=trial,
        kparams=None,
        x=dnn2,
        name_prefix="dense_3",
        activation="tanh",
        units_range=600,
        dropout_rate_range=0.2,
        kernel_initializer=initializer,
    )

    dnn4 = build_dnn(
        trial=trial,
        kparams=None,
        x=dnn3,
        name_prefix="dense_4",
        activation="elu",
        units_range=300,
        dropout_rate_range=0.4,
        kernel_initializer=initializer,
    )

    x = trial_skip_2d_tensors(
        trial=trial,
        layers_list=[dnn1, dnn2, dnn3, dnn4],
        nickname=["dnn1", "dnn2", "dnn3", "dnn4"],
        axis_to_concat=-1,
        strategy="any",
        name_prefix="skip_dnn",
        merge_mode="concat",
        projection_mode="resize",
        verbose=0,
    )

    # —————————————————————————————————— Output —————————————————————————————————— #
    outputs = layers.Dense(
        256,
        activation="softmax",
        name="output",
        kernel_initializer=initializer,
    )(x)

    # —————————————————————————— Set Inputs and Outputs —————————————————————————— #
    model = Model(inputs=(x_lidar_input, x_coord_input), outputs=(outputs,))

    # ———————————————————————————————— Compilation ——————————————————————————————— #
    model.summary() if show_summary else None

    model.compile(
        optimizer=kparams.get_optimizer(trial),
        loss=losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
        jit_compile=False,
    )

    return model

## 5. Optuna Ask

In [ ]:
base_path = "runs/nas_gnn_v2.0_model_param_distribution/"

plot_model_param_distribution(
    lambda trial: build_model(trial=trial, kparams=kparams, show_summary=False),
    bytes_per_param=tf.dtypes.as_dtype(POLICY.variable_dtype).size,
    batch_size=64,
    n_trials=1000,
    fig_save_path=f"{base_path}model_param_distribution.png",
    csv_path=f"{base_path}model_param_distribution.csv",
    logs_dir=f"{base_path}logs/",
    corr_csv_path=f"{base_path}model_param_distribution_corr.csv",
    plot_model_dir=f"{base_path}plots/",
    figsize=(18, 6),
)

[I 2025-08-05 14:24:26,908] A new study created in memory with name: no-name-df5af1ae-4a54-47f7-aea9-e3f220a6715f
  0% [...............................] 0/1000 in ?2025-08-05 14:24:27.133765: E tensorflow/core/util/util.cc:131] oneDNN supports DT_HALF only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


Spektral's GCNConv uses a sparse-dense matmul under the hood.
XLA's GPU JIT compiler does not support that op.
This may cause issues with the GNN layers.
Disable all auto-JIT clustering and auto-JIT compilation.
Call:
os.environ['TF_XLA_FLAGS'] = '--tf_xla_auto_jit=-1'
tf.config.optimizer.set_jit(False)
'jit_compile=False'  # pass into model.compile()


2025-08-05 14:24:27.877192: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: INVALID_ARGUMENT: Expected multiples argument to be a vector of length 4 but got length 3
  0% [.........................] 1/1000 in 1:23:352025-08-05 14:24:32.322528: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: INVALID_ARGUMENT: Expected multiples argument to be a vector of length 4 but got length 3
  0% [...........................] 3/1000 in 50:592025-08-05 14:24:37.350405: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: INVALID_ARGUMENT: Expected multiples argument to be a vector of length 4 but got length 3
  1% [...........................] 7/1000 in 54:222025-08-05 14:24:50.313293: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: INVALID_ARGUMENT: Expected multiples argument to be a vector of length 4 but got length 